Build and write

In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
dbutils.widgets.text("country_code", "CO")
COUNTRY_CODE = dbutils.widgets.get("country_code")
SALT = "dev-only-change-me"   # TODO: move to a Databricks secret scope

# Disclosure terms come from config, lowercased
prof = (spark.table(f"{CATALOG}.config.country_profile")
        .filter(F.col("country_code") == COUNTRY_CODE).first())
lex = [t.lower() for t in prof["disclosure_lexicon"]]
print("lexicon:", lex)

posts    = spark.table(f"{CATALOG}.bronze.posts")
registry = spark.table(f"{CATALOG}.bronze.registry")
venues   = (spark.table(f"{CATALOG}.bronze.venues")
            .select("venue_id", F.col("name").alias("venue_name"),
                    F.col("category").alias("venue_category"),
                    F.col("city").alias("venue_city"), "typical_ticket"))

def pseudo(col):
    return F.sha2(F.concat_ws("|", F.lit(SALT), F.col(col)), 256)

reg_keyed = (registry
    .withColumn("handle_norm", F.lower(F.trim("handle")))
    .withColumn("creator_key", pseudo("creator_id")))

# 1) creators (pseudonymized)
(reg_keyed
    .select("creator_key", "tier", "followers", "declared_income", "is_registered_taxpayer")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.creators"))
keys = reg_keyed.select("handle_norm", "creator_key")
# 2) posts
from pyspark.sql import Window

base = (posts
    .dropDuplicates(["post_id"])
    .withColumn("posted_at", F.to_timestamp("posted_at"))
    .withColumn("handle_norm", F.lower(F.trim("creator_handle")))
    .join(keys, "handle_norm", "left")
    .withColumn("hashtags_norm", F.transform("hashtags", lambda h: F.lower(h)))
    .withColumn("disclosure_tags",
                F.array_intersect("hashtags_norm", F.array(*[F.lit(t) for t in lex])))
    .withColumn("has_disclosure", F.size("disclosure_tags") > 0)
)

cand = base.join(F.broadcast(venues),
                 F.expr("instr(lower(caption), lower(venue_name)) > 0"), "left")

w_all = Window.partitionBy("post_id")
w_rank = w_all.orderBy(
    F.expr("instr(lower(caption), lower(venue_city)) > 0").cast("int").desc(),
    F.length("venue_name").desc(),
    F.col("venue_id"))

silver_posts = (cand
    .withColumn("venue_candidates", F.count("venue_id").over(w_all))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter("rn = 1")
    .drop("rn", "creator_handle", "handle_norm", "hashtags", "_rescued_data"))

(silver_posts.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.posts_enriched"))

Verify

In [0]:
s = spark.table(f"{CATALOG}.silver.posts_enriched")
print("rows:", s.count(), "| distinct post_id:", s.select("post_id").distinct().count())
print("posts without creator_key:", s.filter("creator_key IS NULL").count())
print("posts with a venue match:", s.filter("venue_id IS NOT NULL").count())
print("posts with disclosure tag:", s.filter("has_disclosure").count())
print("columns:", s.columns)
print("handle column present?", "creator_handle" in s.columns)
display(s.filter("venue_id IS NOT NULL").select("caption","disclosure_tags","venue_name","creator_key").limit(5))
print("creators:", spark.table(f"{CATALOG}.silver.creators").count())

In [0]:
# ¿Hay nombres de venue repetidos?
display(venues.groupBy("venue_name").count().filter("count > 1"))

# ¿Qué posts hacen match con más de un venue?
cand = (posts.select("post_id", "caption")
        .join(F.broadcast(venues),
              F.expr("instr(lower(caption), lower(venue_name)) > 0"), "inner"))
dup = (cand.groupBy("post_id", "caption")
       .agg(F.collect_list(F.concat_ws(" | ", "venue_id", "venue_name", "venue_city")).alias("venues"),
            F.count("*").alias("n"))
       .filter("n > 1"))
print("posts con más de un venue candidato:", dup.count())
display(dup.limit(8))